# Anytime D* Concept Demo

This notebook illustrates dynamic replanning with fast-first and improve-over-time behavior.

For pedagogy, we use a simplified weighted-A* schedule after map updates to mimic anytime refinement.

In [ ]:
%matplotlib inline

import heapq
import math

import matplotlib.pyplot as plt
import numpy as np

## 1. Environment and dynamic updates

In [ ]:
H, W = 36, 36
grid0 = np.zeros((H, W), dtype=np.uint8)

grid0[7:30, 11] = 1
grid0[7:30, 25] = 1
grid0[17, 11:26] = 1

# Open doors
grid0[10, 11] = 0
grid0[27, 25] = 0
grid0[17, 19] = 0

start = (3, 3)
goal = (32, 32)

scheduled_updates = {
    1: [(17, 19)],
    2: [(27, 25)],
    3: [(10, 11)]
}

## 2. Weighted A* building block

In [ ]:
DIRS = [(-1, 0), (1, 0), (0, -1), (0, 1)]

def in_bounds(r, c, h, w):
    return 0 <= r < h and 0 <= c < w

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def reconstruct(parent, node):
    path = [node]
    while node in parent:
        node = parent[node]
        path.append(node)
    path.reverse()
    return path

def weighted_astar(grid_map, s, g, epsilon=1.0):
    h, w = grid_map.shape
    open_heap = []
    parent = {}
    g_cost = {s: 0.0}
    closed = set()

    heapq.heappush(open_heap, (epsilon * manhattan(s, g), 0.0, s))
    expansions = 0

    while open_heap:
        _, curr_g, current = heapq.heappop(open_heap)
        if current in closed:
            continue

        closed.add(current)
        expansions += 1

        if current == g:
            return reconstruct(parent, current), g_cost[current], expansions

        for dr, dc in DIRS:
            nr, nc = current[0] + dr, current[1] + dc
            if not in_bounds(nr, nc, h, w):
                continue
            if grid_map[nr, nc] == 1:
                continue

            nxt = (nr, nc)
            ng = curr_g + 1.0
            if ng < g_cost.get(nxt, math.inf):
                g_cost[nxt] = ng
                parent[nxt] = current
                f = ng + epsilon * manhattan(nxt, g)
                heapq.heappush(open_heap, (f, ng, nxt))

    return None, math.inf, expansions

## 3. Anytime pass schedule after each update

In [ ]:
def anytime_passes(grid_map, s, g, eps_schedule):
    records = []
    best = None
    for eps in eps_schedule:
        path, cost, exp = weighted_astar(grid_map, s, g, epsilon=eps)
        records.append({
            'epsilon': eps,
            'path': path,
            'cost': cost,
            'expansions': exp
        })
        if path is not None:
            best = path
    return records, best

eps_schedule = [2.5, 1.8, 1.3, 1.0]

## 4. Run dynamic update episodes

In [ ]:
episodes = []
grid_curr = grid0.copy()

# Episode 0: before updates
recs0, best0 = anytime_passes(grid_curr, start, goal, eps_schedule)
episodes.append(('Initial map', grid_curr.copy(), recs0, best0))

for k in sorted(scheduled_updates.keys()):
    for r, c in scheduled_updates[k]:
        if (r, c) != start and (r, c) != goal:
            grid_curr[r, c] = 1
    recs, best = anytime_passes(grid_curr, start, goal, eps_schedule)
    episodes.append((f'After update {k}', grid_curr.copy(), recs, best))

len(episodes)

## 5. Compare path quality and effort

In [ ]:
for name, _, recs, _ in episodes:
    print(f'\n{name}')
    for r in recs:
        c = r['cost']
        e = r['expansions']
        ok = r['path'] is not None
        print(f'  eps={r["epsilon"]:.1f} | path={ok} | cost={c:.1f} | expansions={e}')

## 6. Visualize best path per episode

In [ ]:
n = len(episodes)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
if n == 1:
    axes = [axes]

for ax, (name, gmap, _, best) in zip(axes, episodes):
    ax.imshow(gmap, cmap='Greys', origin='upper')
    ax.scatter(start[1], start[0], c='limegreen', s=80, marker='o')
    ax.scatter(goal[1], goal[0], c='crimson', s=80, marker='*')

    if best is not None:
        xs = [p[1] for p in best]
        ys = [p[0] for p in best]
        ax.plot(xs, ys, color='deepskyblue', linewidth=2.0)

    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

## 7. Interpretation

This notebook shows the intended behavior pattern of Anytime D*: fast feasible responses after changes, followed by refinement.

A production Anytime D* implementation adds true incremental state reuse and stronger theoretical guarantees.